# Phân tích dữ liệu (EDA): Vietnam Housing Dataset 2024

Mục tiêu: hiểu dữ liệu trước khi huấn luyện Linear Regression dự đoán giá nhà (đơn vị giá: tỷ VNĐ).
Chạy lần lượt từng ô từ trên xuống (Run All).

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Notebook để trong notebooks/ hoặc ở thư mục gốc đều chạy được
DATA = next(p for p in ["../data/gia_nha.csv", "data/gia_nha.csv"] if Path(p).exists())
df = pd.read_csv(DATA)

print("Kích thước dữ liệu:", df.shape)
df.head()

## 1. Tổng quan và dữ liệu thiếu

In [ ]:
missing = pd.DataFrame({
    "Số lượng thiếu": df.isnull().sum(),
    "Tỷ lệ thiếu (%)": (df.isnull().mean() * 100).round(2),
})
print(missing.sort_values("Tỷ lệ thiếu (%)", ascending=False))

print("\nSố dòng trùng lặp:", df.duplicated().sum())
print("\nKiểu dữ liệu:")
print(df.dtypes)

**Nhận xét của bạn:** cột nào thiếu nhiều nhất? Vì sao hai cột hướng vẫn được giữ lại
(gợi ý: trong mô hình, "không ghi hướng" được coi là một nhóm riêng)?

## 2. Giá nhà (biến cần dự đoán)

In [ ]:
print(df["Price"].describe())

plt.figure(figsize=(8, 5))
sns.histplot(df["Price"], bins=30, kde=True)
plt.title("Phân phối giá nhà")
plt.xlabel("Giá nhà (tỷ VNĐ)")
plt.ylabel("Số lượng")
plt.show()

## 3. Các thuộc tính số

In [ ]:
numeric_columns = ["Area", "Frontage", "Access Road", "Floors", "Bedrooms", "Bathrooms"]

plt.figure(figsize=(15, 9))
for i, col in enumerate(numeric_columns):
    plt.subplot(2, 3, i + 1)
    sns.histplot(df[col].dropna(), kde=True)
    plt.title(f"Phân phối {col}")
plt.tight_layout()
plt.show()

## 4. Giá trị ngoại lai

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(y=df["Area"], ax=ax[0]);  ax[0].set_title("Boxplot Area")
sns.boxplot(y=df["Price"], ax=ax[1]); ax[1].set_title("Boxplot Price")
plt.tight_layout()
plt.show()

# Đếm ngoại lai theo quy tắc IQR (ngoài khoảng Q1 - 1.5*IQR đến Q3 + 1.5*IQR)
for col in ["Area", "Price"]:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((df[col] < low) | (df[col] > high)).sum()
    print(f"{col}: Q1={q1}, Q3={q3}, ngưỡng=({low:.1f}; {high:.1f}), "
          f"ngoại lai={n_out} ({n_out / len(df) * 100:.2f}%)")

In [ ]:
# 10 căn có diện tích lớn nhất
df.nlargest(10, "Area")[["Address", "Area", "Bedrooms", "Bathrooms", "Price"]]

**Cách xử lý trong `train_model.py`:** loại các dòng có `Price` hoặc `Area` nằm ngoài khoảng
phân vị 1% đến 99%, vì vài căn diện tích rất lớn ở tỉnh sẽ kéo lệch đường hồi quy.

## 5. Chuẩn hóa địa chỉ

Địa chỉ có dạng `Đường..., Phường/Xã, Quận/Huyện, Tỉnh/Thành`. Vị trí ảnh hưởng đến giá nhiều nhất,
nên tách thành 3 cấp: tỉnh, quận, phường. Dữ liệu còn có một số dòng nhập tay lỗi
("HN", "TPHCM", "quận Tân Bình"...) cần chuẩn hóa. Logic dưới đây giống hệt `train_model.py`.

In [ ]:
CITY_ALIASES = {
    "hn": "Hà Nội",
    "hcm": "Hồ Chí Minh", "tphcm": "Hồ Chí Minh", "tphồchíminh": "Hồ Chí Minh",
}


def clean(s):
    return str(s).strip().rstrip(". ").strip()


def cap(s):
    return s[:1].upper() + s[1:] if isinstance(s, str) else s


def norm_city(s):
    return CITY_ALIASES.get(re.sub(r"[\s.]", "", s.lower()), cap(s))


def norm_district(s):
    # "Quận Long Biên" -> "Long Biên" nhưng giữ "Quận 8"
    return cap(re.sub(r"^(quận|huyện)\s+(?=\D)", "", s, flags=re.IGNORECASE))


parts = df["Address"].astype(str).str.split(",")
df["City"] = parts.apply(lambda p: norm_city(clean(p[-1])) if len(p) >= 2 else np.nan)
df["District"] = parts.apply(lambda p: norm_district(clean(p[-2])) if len(p) >= 2 else np.nan)
df["Ward"] = parts.apply(lambda p: cap(clean(p[-3])) if len(p) >= 3 else np.nan)

# Tỉnh xuất hiện dưới 5 lần gần như chắc chắn là nhập sai
city_counts = df["City"].value_counts()
rare = city_counts[city_counts < 5]
print("Tên tỉnh/thành xuất hiện < 5 lần (bị loại):")
print(rare.index.tolist())

df = df[df["City"].isin(city_counts[city_counts >= 5].index)].copy()
print(f"\nCòn lại: {len(df)} dòng, {df['City'].nunique()} tỉnh/thành, "
      f"{df.groupby(['City', 'District']).ngroups} quận/huyện")
df[["Address", "City", "District", "Ward"]].head()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 6))

top_counts = df["City"].value_counts().head(15)
sns.barplot(x=top_counts.values, y=top_counts.index, ax=ax[0])
ax[0].set_title("15 tỉnh/thành có nhiều tin rao nhất")
ax[0].set_xlabel("Số lượng bất động sản")
ax[0].set_ylabel("")

by_city = df.groupby("City")["Price"].agg(["count", "mean"]).query("count >= 30")["mean"].sort_values(ascending=False)
sns.barplot(x=by_city.head(15).values, y=by_city.head(15).index, ax=ax[1])
ax[1].set_title("Giá trung bình theo tỉnh/thành (từ 30 tin trở lên)")
ax[1].set_xlabel("Giá trung bình (tỷ VNĐ)")
ax[1].set_ylabel("")

plt.tight_layout()
plt.show()

In [ ]:
# Trong cùng một thành phố, giá giữa các quận chênh nhau rất nhiều
fig, ax = plt.subplots(1, 2, figsize=(15, 7))
for a, city in zip(ax, ["Hồ Chí Minh", "Hà Nội"]):
    g = (df[df["City"] == city].groupby("District")["Price"]
         .agg(["count", "mean"]).query("count >= 30")["mean"].sort_values())
    a.barh(g.index, g.values)
    a.set_title(f"Giá trung bình theo quận/huyện: {city}")
    a.set_xlabel("Giá trung bình (tỷ VNĐ)")
plt.tight_layout()
plt.show()

**Nhận xét của bạn:** giá chênh thế nào giữa các tỉnh và giữa các quận trong cùng một thành phố?
Điều này giải thích vì sao mô hình cần tách đến cấp quận, phường chứ không chỉ cấp tỉnh.

## 6. Quan hệ giữa thuộc tính số và giá

In [ ]:
plt.figure(figsize=(15, 9))
for i, col in enumerate(numeric_columns):
    plt.subplot(2, 3, i + 1)
    sns.scatterplot(data=df, x=col, y="Price", alpha=0.3)
    plt.title(f"{col} và Price")
plt.tight_layout()
plt.show()

In [ ]:
correlation = df[numeric_columns + ["Price"]].corr()
print(correlation["Price"].sort_values(ascending=False))

plt.figure(figsize=(9, 6))
sns.heatmap(correlation, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Ma trận tương quan")
plt.tight_layout()
plt.show()

In [ ]:
# Vì sao thêm log(diện tích)? Giá tăng chậm dần theo diện tích, không tăng đều như đường thẳng.
df["LogArea"] = np.log(df["Area"])
print("Tương quan với Price:  Area =", round(df["Area"].corr(df["Price"]), 3),
      "| log(Area) =", round(df["LogArea"].corr(df["Price"]), 3))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=df, x="Area", y="Price", alpha=0.3, ax=ax[0]);    ax[0].set_title("Area và Price")
sns.scatterplot(data=df, x="LogArea", y="Price", alpha=0.3, ax=ax[1]); ax[1].set_title("log(Area) và Price")
plt.tight_layout()
plt.show()

In [ ]:
# Trên toàn quốc, Area tương quan yếu với giá. Thử xét riêng từng thành phố:
rows = []
for city in df["City"].value_counts().head(5).index:
    d = df[df["City"] == city]
    rows.append({"Tỉnh/thành": city, "Số tin": len(d),
                 "Diện tích TB (m²)": round(d["Area"].mean(), 1),
                 "Giá TB (tỷ)": round(d["Price"].mean(), 2),
                 "Tương quan Area-Price": round(d["Area"].corr(d["Price"]), 3)})
print("Toàn quốc: tương quan Area-Price =", round(df["Area"].corr(df["Price"]), 3))
pd.DataFrame(rows)

**Nhận xét của bạn:** thuộc tính nào tương quan mạnh nhất với giá? Vì sao `Area` lại tương quan yếu
trên toàn quốc nhưng mạnh hơn khi xét từng thành phố? (gợi ý: nhà rộng ở tỉnh vẫn rẻ hơn nhà nhỏ
ở trung tâm, nên vị trí che mờ ảnh hưởng của diện tích)

## 7. Thuộc tính phân loại

In [ ]:
categorical_columns = ["Legal status", "Furniture state", "House direction", "Balcony direction"]

for col in categorical_columns:
    print("\n" + "=" * 50)
    print(col)
    print(df[col].value_counts(dropna=False))

In [ ]:
# Giá trung bình theo từng nhóm. "Không rõ" là các dòng không ghi thông tin này.
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, col in zip(axes.ravel(), categorical_columns):
    data = df.assign(**{col: df[col].fillna("Không rõ")})
    order = data.groupby(col)["Price"].mean().sort_values(ascending=False).index
    sns.barplot(data=data, x=col, y="Price", order=order, errorbar=None, ax=ax)
    ax.set_title(f"Giá trung bình theo {col}")
    ax.set_ylabel("Giá trung bình (tỷ VNĐ)")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## 8. Kết luận EDA (tự viết)

- Dữ liệu: ... dòng, ... cột, cột thiếu nhiều nhất là ...
- Giá nhà: phân phối ..., nằm trong khoảng ... đến ... tỷ
- Ngoại lai: ... (xử lý bằng cách ...)
- Yếu tố ảnh hưởng đến giá nhiều nhất: ...
- Hạn chế của dữ liệu: ...